# Audience segmentation: disorganized prospect exports to actionable clusters

This notebook generates 500 prospects and 200 deal-history rows with the repository's real generators, injects spreadsheet defects, restores the exact source contract, then runs the real TF-IDF, UMAP, K-Means, DBSCAN, profiling, visualization, and reporting pipeline before executing all tests.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run this notebook from the repository root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "notebooks" / "data"
ARTIFACT_DIR = ROOT / "notebooks" / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository: {ROOT.name}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
import pandas as pd

from generate_data import _generate_deals, _generate_prospects

canonical_prospects = pd.DataFrame(_generate_prospects(500, seed=42))
canonical_deals = pd.DataFrame(
    _generate_deals(canonical_prospects.to_dict("records"), 200, seed=43)
)

prospects = canonical_prospects.copy().astype(object)
for column in ["annual_revenue", "current_ad_spend_estimate"]:
    prospects[column] = prospects[column].map(lambda value: f" ${int(value):,} ")
prospects["employee_count"] = prospects["employee_count"].map(lambda value: f"{int(value):,}")
for column in [
    "company_name",
    "industry",
    "website_description",
    "recent_campaigns",
    "target_demographics",
]:
    prospects[column] = prospects[column].map(lambda value: f"  {str(value)}  ")
prospects["source_note"] = "manual CRM export"
bad_prospect = prospects.iloc[[0]].copy()
bad_prospect["company_name"] = ""
prospects = pd.concat([prospects, prospects.iloc[:8], bad_prospect], ignore_index=True).sample(
    frac=1, random_state=42
)

deals = canonical_deals.copy().astype(object)
deals["deal_value"] = deals["deal_value"].map(lambda value: f"${int(value):,}")
deals["sales_cycle_days"] = deals["sales_cycle_days"].map(str)
deals["company_name"] = deals["company_name"].map(lambda value: f" {value} ")
deals["outcome"] = deals["outcome"].map(lambda value: f" {str(value).upper()} ")
deals["channels_used"] = deals["channels_used"].map(lambda value: f" {value} ")
deals["source_note"] = "manual sales export"
deals = pd.concat([deals, deals.iloc[:5]], ignore_index=True).sample(frac=1, random_state=43)

messy_prospects_path = DATA_DIR / "messy_prospects.csv"
messy_deals_path = DATA_DIR / "messy_deal_history.csv"
prospects.to_csv(messy_prospects_path, index=False)
deals.to_csv(messy_deals_path, index=False)
print({"prospects": len(prospects), "deals": len(deals)})
prospects.head(3)

In [ ]:
clean_prospects = prospects.drop(columns=["source_note"]).copy()
for column in ["annual_revenue", "current_ad_spend_estimate", "employee_count"]:
    clean_prospects[column] = pd.to_numeric(
        clean_prospects[column].astype(str).str.replace(r"[$,\s]", "", regex=True), errors="coerce"
    )
for column in [
    "company_name",
    "industry",
    "website_description",
    "recent_campaigns",
    "target_demographics",
]:
    clean_prospects[column] = clean_prospects[column].astype(str).str.strip()
clean_prospects = (
    clean_prospects[clean_prospects["company_name"].ne("")].dropna().drop_duplicates("company_name")
)
for column in [
    "annual_revenue",
    "current_ad_spend_estimate",
    "employee_count",
    "social_presence_score",
]:
    clean_prospects[column] = clean_prospects[column].astype(int)
clean_prospects = clean_prospects[canonical_prospects.columns].reset_index(drop=True)

clean_deals = deals.drop(columns=["source_note"]).copy()
clean_deals["deal_value"] = pd.to_numeric(
    clean_deals["deal_value"].astype(str).str.replace(r"[$,]", "", regex=True)
)
clean_deals["sales_cycle_days"] = pd.to_numeric(clean_deals["sales_cycle_days"])
clean_deals["company_name"] = clean_deals["company_name"].astype(str).str.strip()
clean_deals["outcome"] = clean_deals["outcome"].astype(str).str.strip().str.lower()
clean_deals["channels_used"] = clean_deals["channels_used"].astype(str).str.strip()
clean_deals = clean_deals.drop_duplicates().astype({"deal_value": int, "sales_cycle_days": int})
clean_deals = clean_deals[canonical_deals.columns].reset_index(drop=True)

pd.testing.assert_frame_equal(
    clean_prospects.sort_values("company_name").reset_index(drop=True),
    canonical_prospects.sort_values("company_name").reset_index(drop=True),
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    clean_deals.sort_values(list(canonical_deals.columns)).reset_index(drop=True),
    canonical_deals.sort_values(list(canonical_deals.columns)).reset_index(drop=True),
    check_dtype=False,
)
clean_prospects_path = DATA_DIR / "cleaned_prospects.csv"
clean_deals_path = DATA_DIR / "cleaned_deal_history.csv"
clean_prospects.to_csv(clean_prospects_path, index=False)
clean_deals.to_csv(clean_deals_path, index=False)
print("Recovered 500 prospects and 200 deal-history rows exactly.")

In [ ]:
from segment import run_pipeline

output = ARTIFACT_DIR / "segmentation"
result = run_pipeline(
    input_path=str(clean_prospects_path),
    deals_path=str(clean_deals_path),
    output_dir=str(output),
    n_clusters="auto",
    use_umap=True,
    run_dbscan_comparison=True,
)
required = {"segments.csv", "segment_profiles.json", "visualization.html", "report.md"}
assert required.issubset({path.name for path in output.iterdir()})
assert len(result["labels"]) == 500
assert result["profiles"]
print(
    {
        "optimal_k": result["optimal_k"],
        "profiles": len(result["profiles"]),
        "dbscan_profiles": len(result["dbscan_profiles"]),
    }
)

In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(completed.stdout)
print(completed.stderr)
assert completed.returncode == 0, "Repository tests failed"
print("VALIDATED: the repository test suite passes in this UV environment.")